# Diabetes Prediction using Machine Learning

I use the supplied patient data to train a model that predicts `Outcome` (0 = no diabetes, 1 = diabetes). The notebook walks through the data, explains the preprocessing choices, and evaluates a Logistic Regression model.

## 1. Import libraries and load the dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
data_path = Path("data/diabetes.csv")
if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_path}. Run this notebook from the project directory."
    )

data = pd.read_csv(data_path)
data.head()

## 2. Explore the data

The dataset has 768 patient records and eight numerical predictors, including glucose, BMI, and age. `Outcome` is the value we want to predict. There are no categorical features to encode.

In [ ]:
print(f"Dataset shape: {data.shape}")
print("\nColumns:")
print(data.columns.tolist())
print("\nData types:")
print(data.dtypes)
print("\nMissing values by column:")
print(data.isna().sum())

In [ ]:
data.describe().T

In [ ]:
outcome_counts = data["Outcome"].value_counts().sort_index()
print("Outcome counts (0 = no diabetes, 1 = diabetes):")
print(outcome_counts)

outcome_counts.plot(kind="bar", color=["steelblue", "darkorange"])
plt.title("Distribution of Diabetes Outcome")
plt.xlabel("Outcome")
plt.ylabel("Number of patients")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

The CSV has no blank cells, but several measurement columns contain zeros that likely stand for values that were not recorded. I treat zeros in Glucose, BloodPressure, SkinThickness, Insulin, and BMI as missing. Zero pregnancies is possible, so I leave that column unchanged. The pipeline fills missing values using medians learned from the training data.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Zero values in measurement columns before preprocessing:")
print((data[zero_as_missing] == 0).sum())

## 3. Prepare features and split the data

I use the eight patient attributes as `X` and `Outcome` as `y`. The stratified 80/20 split keeps the outcome proportions similar in both sets, and the fixed random seed makes the results reproducible.

In [ ]:
X = data.drop(columns="Outcome").copy()
y = data["Outcome"].copy()
X[zero_as_missing] = X[zero_as_missing].replace(0, np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Features: {X.columns.tolist()}")
print(f"Target: {y.name}")
print(f"Training rows: {len(X_train)}; testing rows: {len(X_test)}")
print(f"Training target counts:\n{y_train.value_counts().sort_index()}")
print(f"\nTesting target counts:\n{y_test.value_counts().sort_index()}")

## 4. Build and train the model

I chose Logistic Regression because this is a binary classification problem and the model provides a straightforward baseline. Since the features use different units, I scale them before fitting. The pipeline also fills in the missing measurements; keeping both steps in the pipeline ensures they are learned from the training set only.

In [ ]:
model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model.fit(X_train, y_train)

In [ ]:
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)
test_precision = precision_score(y_test, test_predictions, zero_division=0)
test_recall = recall_score(y_test, test_predictions, zero_division=0)
test_f1 = f1_score(y_test, test_predictions, zero_division=0)

print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Testing accuracy:  {test_accuracy:.3f}")
print(f"Testing precision: {test_precision:.3f}")
print(f"Testing recall:    {test_recall:.3f}")
print(f"Testing F1-score:  {test_f1:.3f}")

## 5. Evaluate and analyze performance

Accuracy gives the share of correct predictions. Precision and recall help show how the model handles the positive class, and the F1-score combines the two. The confusion matrix shows the actual labels in rows and predictions in columns.

In [ ]:
print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=["No diabetes (0)", "Diabetes (1)"],
    zero_division=0,
))

cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])
print("Confusion matrix (rows = actual, columns = predicted):")
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm, cmap="Blues")
ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["No diabetes", "Diabetes"],
    yticklabels=["No diabetes", "Diabetes"],
    xlabel="Predicted label",
    ylabel="Actual label",
    title="Test-set confusion matrix",
)
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, cm[row, col], ha="center", va="center")
fig.colorbar(image, ax=ax, label="Number of patients")
fig.tight_layout()
plt.show()

In [ ]:
accuracy_gap = train_accuracy - test_accuracy
print(f"Training vs. testing accuracy gap: {accuracy_gap:+.3f}")
if accuracy_gap > 0.10:
    print("Training accuracy is noticeably higher than test accuracy, which may indicate overfitting.")
elif accuracy_gap < -0.10:
    print("Test accuracy is higher than training accuracy; this can happen due to differences in the split.")
else:
    print("Training and test accuracy are fairly close for this split.")

print(
    f"The model identifies {test_recall:.1%} of diabetes-positive patients in the test set "
    f"(recall = {test_recall:.3f})."
)
print(
    "Accuracy alone does not show how many positive cases were missed, so I also consider "
    "precision, recall, and the confusion matrix."
)

## 6. Conclusion and limitations

The model's training and test scores give a first look at how well Logistic Regression fits this dataset. Test results come from one split, so they may change with a different sample. Treating measurement zeros as missing is an assumption, and the dataset is small. A stronger follow-up would compare other models and evaluate them on additional data. This exercise is for learning and is not a medical diagnostic tool.